# Hybrid Models: ARIMA/SARIMA + SVR/MLP

This notebook implements four hybrid time-series forecasting models:
- **ARIMA + SVR**: ARIMA captures the linear component; SVR corrects the nonlinear residuals.
- **ARIMA + MLP**: Same as above but using a two-layer PyTorch MLP for residual correction.
- **SARIMA + SVR**: SARIMA (seasonal) as linear component + SVR residual correction.
- **SARIMA + MLP**: SARIMA + MLP residual correction.

The final prediction is:
```
final_pred[t] = linear_model_pred[t] + nonlinear_model(residuals[t-k : t])
```
Pre-computed ARIMA/SARIMA predictions are loaded from `datasets/arima_data.xlsx` and
`datasets/sarima_data.xlsx`. The SVR/MLP is trained on lagged residuals (lookback = 10)
with strict split-boundary enforcement to prevent data leakage.

Results (including training time and memory usage) are saved to `results/hybrid_models_results.csv`.

In [1]:
# ============================================================
# CELL 1 – IMPORTS & CONFIGURATION
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from openpyxl import load_workbook
import time
import tracemalloc
import psutil
import os
import warnings
from typing import Tuple, Dict

warnings.filterwarnings("ignore")

# ── Device ──────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Experiment constants ─────────────────────────────────────
LOOKBACK   = 10
NUM_RUNS   = 15
MAX_EPOCHS = 200
PATIENCE   = 30
BATCH_SIZE = 32

# ── SVR hyperparameters ──────────────────────────────────────
SVR_CONFIG = {
    "kernel" : "rbf",
    "C"      : 100,
    "gamma"  : "scale",
    "epsilon": 0.1,
}

# ── MLP hyperparameters ──────────────────────────────────────
MLP_CONFIG = {
    "hidden_size": 64,
    "dropout"    : 0.1,
    "lr"         : 0.001,
}

# ── Datasets (same as LSTM experiments) ─────────────────────
DATASETS = [
    "CARSALES", "Electricity", "GAS", "LAKEERIE",
    "Nordic", "PIGS", "POLLUTION", "REDWINE", "SUNSPOT", "B1H",
]

print(f"Device       : {device}")
print(f"Lookback     : {LOOKBACK}")
print(f"Runs         : {NUM_RUNS}")
print(f"Max epochs   : {MAX_EPOCHS}  (patience={PATIENCE})")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Datasets     : {DATASETS}")

Device       : cpu
Lookback     : 10
Runs         : 15
Max epochs   : 200  (patience=30)
Batch size   : 32
Datasets     : ['CARSALES', 'Electricity', 'GAS', 'LAKEERIE', 'Nordic', 'PIGS', 'POLLUTION', 'REDWINE', 'SUNSPOT', 'B1H']


In [2]:
# ============================================================
# CELL 2 – REPRODUCIBILITY & METRICS
# ============================================================

def set_seed(seed: int = 42) -> None:
    """Set all random seeds for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """Compute MSE, RMSE, and R² between true and predicted arrays."""
    if len(y_true) == 0:
        return {"mse": float("nan"), "rmse": float("nan"), "r2": float("nan")}
    mse    = float(np.mean((y_true - y_pred) ** 2))
    rmse   = float(np.sqrt(mse))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2     = float(1.0 - ss_res / ss_tot) if ss_tot != 0 else 0.0
    return {"mse": mse, "rmse": rmse, "r2": r2}


print("Reproducibility & metrics helpers loaded.")

Reproducibility & metrics helpers loaded.


In [3]:
# ============================================================
# CELL 3 – DATA LOADING
# ============================================================

def _normalize_split(label: str) -> str:
    """Normalize a split label to one of: 'Treinamento', 'Validacao', 'Teste'."""
    lc = label.strip().lower()
    if "trein" in lc:
        return "Treinamento"
    if "valid" in lc or "valida" in lc:
        return "Validacao"
    if "teste" in lc or "test" in lc:
        return "Teste"
    raise ValueError(f"Unknown split label: '{label}'")


def load_arima_data(
    dataset_name: str,
    excel_path: str = "./datasets/arima_data.xlsx",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load (serie, arima_pred, split_labels) from arima_data.xlsx.

    Column layout (located by header name):
        index | SERIE | SPLIT | ARIMA | MLP | SVR | RBF
    """
    wb = load_workbook(excel_path, read_only=True, data_only=True)
    if dataset_name not in wb.sheetnames:
        raise ValueError(f"Dataset '{dataset_name}' not found in {excel_path}")
    ws = wb[dataset_name]

    raw_header = next(ws.iter_rows(min_row=1, max_row=1, values_only=True))
    header = [
        str(h).strip().upper() if h is not None else f"COL_{i}"
        for i, h in enumerate(raw_header)
    ]

    serie_col = next(i for i, h in enumerate(header) if "SERIE" in h)
    split_col = next(i for i, h in enumerate(header) if "SPLIT" in h or "CONJUNTO" in h)
    arima_col = next(i for i, h in enumerate(header) if "ARIMA" in h)

    min_cols = max(serie_col, split_col, arima_col) + 1
    rows = [
        r for r in ws.iter_rows(min_row=2, values_only=True)
        if len(r) >= min_cols
        and r[serie_col] is not None
        and r[arima_col] is not None
    ]

    serie      = np.array([float(r[serie_col]) for r in rows], dtype=np.float64)
    arima_pred = np.array([float(r[arima_col]) for r in rows], dtype=np.float64)
    splits     = np.array([_normalize_split(str(r[split_col])) for r in rows])

    print(
        f"[ARIMA/{dataset_name}] n={len(serie)} "
        f"| Train={np.sum(splits=='Treinamento')} "
        f"| Val={np.sum(splits=='Validacao')} "
        f"| Test={np.sum(splits=='Teste')}"
    )
    return serie, arima_pred, splits


def load_sarima_data(
    dataset_name: str,
    excel_path: str = "./datasets/sarima_data.xlsx",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load (serie, sarima_pred, split_labels) from sarima_data.xlsx.

    Column layout (located by header name):
        index | SPLIT | SARIMA | MLP | SVR | RBF | SERIE
    Split labels are uppercase in this file; _normalize_split handles them.
    """
    wb = load_workbook(excel_path, read_only=True, data_only=True)
    if dataset_name not in wb.sheetnames:
        raise ValueError(f"Dataset '{dataset_name}' not found in {excel_path}")
    ws = wb[dataset_name]

    raw_header = next(ws.iter_rows(min_row=1, max_row=1, values_only=True))
    header = [
        str(h).strip().upper() if h is not None else f"COL_{i}"
        for i, h in enumerate(raw_header)
    ]

    serie_col  = next(i for i, h in enumerate(header) if "SERIE"  in h)
    split_col  = next(i for i, h in enumerate(header) if "SPLIT"  in h or "CONJUNTO" in h)
    sarima_col = next(i for i, h in enumerate(header) if "SARIMA" in h)

    min_cols = max(serie_col, split_col, sarima_col) + 1
    rows = [
        r for r in ws.iter_rows(min_row=2, values_only=True)
        if len(r) >= min_cols
        and r[serie_col] is not None
        and r[sarima_col] is not None
    ]

    serie       = np.array([float(r[serie_col])  for r in rows], dtype=np.float64)
    sarima_pred = np.array([float(r[sarima_col]) for r in rows], dtype=np.float64)
    splits      = np.array([_normalize_split(str(r[split_col])) for r in rows])

    print(
        f"[SARIMA/{dataset_name}] n={len(serie)} "
        f"| Train={np.sum(splits=='Treinamento')} "
        f"| Val={np.sum(splits=='Validacao')} "
        f"| Test={np.sum(splits=='Teste')}"
    )
    return serie, sarima_pred, splits


print("Data-loading helpers loaded.")

Data-loading helpers loaded.


In [4]:
# ============================================================
# CELL 4 – RESIDUAL SEQUENCE CREATION (NO DATA LEAKAGE)
# ============================================================

def create_residual_sequences(
    serie       : np.ndarray,
    linear_pred : np.ndarray,
    split_labels: np.ndarray,
    lookback    : int = 10,
) -> Dict[str, np.ndarray]:
    """
    Build lagged-residual feature matrices for the nonlinear correction model.

    For every position i (>= lookback):
      residuals[i] = serie[i] - linear_pred[i]
      X[i]         = residuals[i-lookback : i]   <- lookback-length feature vector
      y[i]         = residuals[i]                <- regression target

    Leakage guard
    -------------
    The entire lookback window must lie within the same split as position i.
    Sequences whose window crosses a split boundary are discarded.

    Returns
    -------
    Dict with keys:
      X_train  / X_val  / X_test   : feature matrices   (n x lookback)
      y_train  / y_val  / y_test   : residual targets    (n,)
      lp_train / lp_val / lp_test  : aligned linear predictions (n,)
      yt_train / yt_val / yt_test  : aligned actual series values (n,)
    """
    residuals = (serie - linear_pred).astype(np.float64)

    buckets: Dict[str, list] = {
        k: [] for k in [
            "X_train",  "X_val",  "X_test",
            "y_train",  "y_val",  "y_test",
            "lp_train", "lp_val", "lp_test",
            "yt_train", "yt_val", "yt_test",
        ]
    }

    split_map = {
        "Treinamento": ("X_train",  "y_train",  "lp_train", "yt_train"),
        "Validacao"  : ("X_val",    "y_val",    "lp_val",   "yt_val"),
        "Teste"      : ("X_test",   "y_test",   "lp_test",  "yt_test"),
    }

    for i in range(lookback, len(serie)):
        target_split = split_labels[i]
        if target_split not in split_map:
            continue
        # Enforce no-leakage: all look-back steps must share the same split
        if not np.all(split_labels[i - lookback : i] == target_split):
            continue

        xk, yk, lpk, ytk = split_map[target_split]
        buckets[xk].append(residuals[i - lookback : i].copy())
        buckets[yk].append(residuals[i])
        buckets[lpk].append(linear_pred[i])
        buckets[ytk].append(serie[i])

    # Convert to numpy arrays
    result: Dict[str, np.ndarray] = {}
    for k, lst in buckets.items():
        if k.startswith("X"):
            result[k] = (
                np.array(lst, dtype=np.float64) if lst
                else np.empty((0, lookback), dtype=np.float64)
            )
        else:
            result[k] = np.array(lst, dtype=np.float64)

    return result


print("Residual-sequence helper loaded.")

Residual-sequence helper loaded.


In [5]:
# ============================================================
# CELL 5 – SVR MODEL
# ============================================================

def train_svr(
    X_train: np.ndarray,
    y_train: np.ndarray,
    config : Dict = SVR_CONFIG,
) -> Tuple["SVR", StandardScaler]:
    """
    Fit a StandardScaler (trained on X_train) and an SVR model on residuals.

    The scaler is fit only on training data and then applied to val/test features,
    preventing feature-scale leakage.

    SVR is deterministic: all NUM_RUNS runs for a given dataset produce
    identical results (included for format consistency with the MLP / LSTM
    experiments so that Wilcoxon / Nemenyi tests can be applied uniformly).

    Returns: (fitted_svr, fitted_scaler)
    """
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)
    svr = SVR(
        kernel  = config["kernel"],
        C       = config["C"],
        gamma   = config["gamma"],
        epsilon = config["epsilon"],
    )
    svr.fit(X_scaled, y_train)
    return svr, scaler


def predict_svr(
    svr    : "SVR",
    scaler : StandardScaler,
    X      : np.ndarray,
) -> np.ndarray:
    """Apply a trained SVR to new (unscaled) feature vectors."""
    if len(X) == 0:
        return np.array([], dtype=np.float64)
    return svr.predict(scaler.transform(X)).astype(np.float64)


print("SVR helpers loaded.")

SVR helpers loaded.


In [6]:
# ============================================================
# CELL 6 – MLP MODEL (PyTorch)
# ============================================================

# ── Dataset wrapper ──────────────────────────────────────────
class ResidualDataset(Dataset):
    """Thin PyTorch Dataset wrapping numpy residual arrays."""

    def __init__(self, X: np.ndarray, y: np.ndarray) -> None:
        self.X = torch.FloatTensor(X.astype(np.float32))
        self.y = torch.FloatTensor(y.astype(np.float32))

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]


# ── Model architecture ───────────────────────────────────────
class MLPModel(nn.Module):
    """
    Two-hidden-layer feedforward MLP for residual regression.

    Architecture: input_size → hidden_size → hidden_size → 1
    Activation:   ReLU after each hidden layer, with optional Dropout.
    """

    def __init__(
        self,
        input_size : int   = 10,
        hidden_size: int   = 64,
        dropout    : float = 0.1,
    ) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)


# ── Early stopping ───────────────────────────────────────────
class EarlyStopping:
    """
    Monitors validation loss and halts training when there is no improvement
    for `patience` consecutive epochs.  Saves and restores the best checkpoint.
    """

    def __init__(self, patience: int = 10, min_delta: float = 0.0) -> None:
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = float("inf")
        self.best_state : Dict = {}
        self.best_epoch = 0

    def __call__(self, val_loss: float, model: nn.Module, epoch: int) -> bool:
        """Return True when training should stop."""
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            self.best_epoch = epoch
            self.counter    = 0
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model: nn.Module) -> None:
        """Load the checkpoint with the best validation loss."""
        if self.best_state:
            model.load_state_dict(self.best_state)


# ── Training loop ────────────────────────────────────────────
def train_mlp(
    X_train   : np.ndarray,
    y_train   : np.ndarray,
    X_val     : np.ndarray,
    y_val     : np.ndarray,
    config    : Dict = MLP_CONFIG,
    max_epochs: int  = MAX_EPOCHS,
    patience  : int  = PATIENCE,
) -> Tuple[MLPModel, Dict]:
    """
    Train a two-layer MLP with Adam optimiser and early stopping on val MSE.

    Mirrors the LSTM training loop in ltsm.ipynb:
      - gradient clipping (max_norm=1.0)
      - best-checkpoint restoration at the end

    Returns: (best_model, {"best_epoch": int})
    """
    model = MLPModel(
        input_size  = X_train.shape[1],
        hidden_size = config["hidden_size"],
        dropout     = config["dropout"],
    ).to(device)

    criterion  = nn.MSELoss()
    optimizer  = torch.optim.Adam(model.parameters(), lr=config["lr"])
    early_stop = EarlyStopping(patience=patience)

    train_loader = DataLoader(
        ResidualDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True
    )
    val_loader = DataLoader(
        ResidualDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False
    )

    for epoch in range(max_epochs):
        # ── Training step
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # ── Validation step
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                val_loss += criterion(model(Xb), yb).item() * len(Xb)
        val_loss /= max(len(val_loader.dataset), 1)

        if early_stop(val_loss, model, epoch + 1):
            break

    early_stop.restore(model)
    return model, {"best_epoch": early_stop.best_epoch}


# ── Inference ────────────────────────────────────────────────
def predict_mlp(model: MLPModel, X: np.ndarray) -> np.ndarray:
    """Run a forward pass and return predictions as a float64 numpy array."""
    if len(X) == 0:
        return np.array([], dtype=np.float64)
    model.eval()
    with torch.no_grad():
        preds = model(torch.FloatTensor(X.astype(np.float32)).to(device))
    return preds.cpu().numpy().astype(np.float64)


print("MLP helpers loaded.")

MLP helpers loaded.


In [7]:
# ============================================================
# CELL 7 – TRAINING TIME & MEMORY MEASUREMENT
# ============================================================

def measure_training(train_fn, *args, **kwargs) -> Tuple:
    """
    Execute train_fn(*args, **kwargs) while recording four profiling signals.

    Signals
    -------
    training_time_sec
        Wall-clock duration of the training call (time.perf_counter).

    peak_memory_mb  (RSS delta)
        Growth in the process Resident Set Size during training, measured
        with psutil.  This captures memory allocated by native C/C++
        extensions (e.g. libsvm used by sklearn SVR) that live outside
        Python's managed heap.  Clamped to 0 if RSS shrinks (GC activity).

    peak_python_memory_mb  (tracemalloc peak)
        Peak allocation on the Python-managed heap during training,
        measured with the stdlib tracemalloc module.  Accurately reflects
        PyTorch tensor buffers and NumPy array allocations.

    gpu_memory_mb
        Peak GPU memory allocated during training
        (torch.cuda.max_memory_allocated).  Returns 0.0 on CPU-only runs.

    Returns
    -------
    (result, training_time_sec, peak_memory_mb,
     peak_python_memory_mb, gpu_memory_mb)
    """
    proc = psutil.Process(os.getpid())

    # Baseline RSS before training
    rss_before = proc.memory_info().rss

    # Reset GPU peak counter (no-op on CPU-only machines)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Start Python-heap tracker
    tracemalloc.start()

    # ── Training ─────────────────────────────────────────────
    t0     = time.perf_counter()
    result = train_fn(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    # ─────────────────────────────────────────────────────────

    # Capture Python-heap peak BEFORE stopping tracemalloc
    _, peak_python_bytes = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Capture RSS after training
    rss_after = proc.memory_info().rss

    # Capture GPU peak
    gpu_memory_mb = (
        torch.cuda.max_memory_allocated() / 1e6
        if torch.cuda.is_available() else 0.0
    )

    peak_memory_mb        = max(0.0, (rss_after - rss_before)) / 1e6
    peak_python_memory_mb = peak_python_bytes / 1e6

    return result, elapsed, peak_memory_mb, peak_python_memory_mb, gpu_memory_mb


print("Timing & memory-measurement helper loaded.")

Timing & memory-measurement helper loaded.


In [8]:
# ============================================================
# CELL 8 – RESULTS SAVING
# ============================================================

def save_results_to_csv(
    results_data: Dict,
    filename    : str = "results/hybrid_models_results.csv",
) -> None:
    """
    Append a single experiment-result dict to a CSV file.

    Creates the file with a header row on the first call;
    appends without header on subsequent calls.

    Output columns
    --------------
    experiment_num          Run index (1 – NUM_RUNS)
    dataset                 Dataset name
    method                  ARIMA_SVR | ARIMA_MLP | SARIMA_SVR | SARIMA_MLP
    lookback                Feature window size (10)
    best_epoch              Best epoch for MLP; 1 for SVR
    training_time_sec       Wall-clock training duration (seconds)
    peak_memory_mb          Peak RSS growth during training (MB)
    peak_python_memory_mb   Peak Python-heap allocation via tracemalloc (MB)
    gpu_memory_mb           Peak GPU memory (MB); 0.0 on CPU
    mse_train / rmse_train / r2_train
    mse_val   / rmse_val   / r2_val
    mse_test  / rmse_test  / r2_test
    """
    dir_name = os.path.dirname(filename)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)

    df = pd.DataFrame([results_data])
    if os.path.exists(filename):
        df.to_csv(filename, mode="a", header=False, index=False)
    else:
        df.to_csv(filename, mode="w", header=True, index=False)


print("Results-saving helper loaded.")

Results-saving helper loaded.


In [9]:
# ============================================================
# CELL 9 – EXPERIMENT PIPELINE
# ============================================================

def run_single_experiment(
    dataset_name : str,
    run_num      : int,
    model_type   : str,          # "ARIMA_SVR" | "ARIMA_MLP" | "SARIMA_SVR" | "SARIMA_MLP"
    serie        : np.ndarray,
    linear_pred  : np.ndarray,
    split_labels : np.ndarray,
) -> Dict:
    """
    Run one hybrid-model experiment.

    Pipeline
    --------
    1. Set random seed (42 + run_num) for reproducibility.
    2. Build lagged-residual sequences with no-leakage enforcement.
    3. Train SVR or MLP on training residuals while capturing
       wall-clock time, RSS memory delta, Python-heap peak, GPU peak.
    4. Predict residuals on all splits using the trained model.
    5. Combine: final_pred = linear_pred + predicted_residual.
    6. Compute MSE, RMSE, R² for train / val / test splits.
    7. Return a flat dict ready for CSV serialisation.
    """
    set_seed(42 + run_num)

    # ── 1. Build sequences ───────────────────────────────────
    seqs = create_residual_sequences(serie, linear_pred, split_labels, LOOKBACK)

    X_train, y_train = seqs["X_train"], seqs["y_train"]
    X_val,   y_val   = seqs["X_val"],   seqs["y_val"]

    lp_train = seqs["lp_train"]
    lp_val   = seqs["lp_val"]
    lp_test  = seqs["lp_test"]
    yt_train = seqs["yt_train"]
    yt_val   = seqs["yt_val"]
    yt_test  = seqs["yt_test"]

    if len(X_train) < 5:
        raise ValueError(f"Insufficient training sequences: {len(X_train)}")

    # ── 2. Train with time + memory measurement ──────────────
    is_svr     = model_type.endswith("SVR")
    best_epoch = 1     # default for SVR (no iterative epochs)

    if is_svr:
        def _train():
            return train_svr(X_train, y_train)

        (trained_svr, scaler), t_sec, mem_mb, py_mem_mb, gpu_mb = \
            measure_training(_train)

    else:  # MLP
        # If val split produced no usable sequences (edge case for very short
        # series), fall back to a held-out fraction of the training set so
        # that early stopping still has a signal.
        if len(X_val) > 0:
            X_v, y_v = X_val, y_val
        else:
            split_idx = max(1, int(0.8 * len(X_train)))
            X_v, y_v  = X_train[split_idx:], y_train[split_idx:]
            X_train   = X_train[:split_idx]
            y_train   = y_train[:split_idx]

        def _train():
            return train_mlp(X_train, y_train, X_v, y_v)

        (trained_mlp, history), t_sec, mem_mb, py_mem_mb, gpu_mb = \
            measure_training(_train)
        best_epoch = history["best_epoch"]

    # ── 3. Predict residuals on all splits ───────────────────
    if is_svr:
        res_train = predict_svr(trained_svr, scaler, seqs["X_train"])
        res_val   = predict_svr(trained_svr, scaler, seqs["X_val"])
        res_test  = predict_svr(trained_svr, scaler, seqs["X_test"])
    else:
        res_train = predict_mlp(trained_mlp, seqs["X_train"])
        res_val   = predict_mlp(trained_mlp, seqs["X_val"])
        res_test  = predict_mlp(trained_mlp, seqs["X_test"])

    # ── 4. Combine: final = linear_pred + predicted_residual ─
    final_train = lp_train + res_train
    final_val   = lp_val   + res_val   if len(res_val)  > 0 else np.array([])
    final_test  = lp_test  + res_test  if len(res_test) > 0 else np.array([])

    # ── 5. Metrics ───────────────────────────────────────────
    train_m = calculate_metrics(yt_train, final_train)
    val_m   = calculate_metrics(yt_val,   final_val)
    test_m  = calculate_metrics(yt_test,  final_test)

    return {
        "experiment_num"        : run_num,
        "dataset"               : dataset_name,
        "method"                : model_type,
        "lookback"              : LOOKBACK,
        "best_epoch"            : best_epoch,
        "training_time_sec"     : t_sec,
        "peak_memory_mb"        : mem_mb,
        "peak_python_memory_mb" : py_mem_mb,
        "gpu_memory_mb"         : gpu_mb,
        "mse_train"             : train_m["mse"],
        "rmse_train"            : train_m["rmse"],
        "r2_train"              : train_m["r2"],
        "mse_val"               : val_m["mse"],
        "rmse_val"              : val_m["rmse"],
        "r2_val"                : val_m["r2"],
        "mse_test"              : test_m["mse"],
        "rmse_test"             : test_m["rmse"],
        "r2_test"               : test_m["r2"],
    }


def run_dataset_experiments(
    dataset_name: str,
    model_type  : str,
    output_file : str = "results/hybrid_models_results.csv",
) -> list:
    """
    Run NUM_RUNS experiments for one (dataset, model_type) combination.

    Loads the appropriate linear-model predictions (ARIMA or SARIMA),
    loops over runs, prints per-run and summary stats, saves each row to CSV.

    Returns a list of result dicts.
    """
    # Load the correct linear predictions
    if model_type.startswith("ARIMA"):
        serie, linear_pred, split_labels = load_arima_data(dataset_name)
    else:
        serie, linear_pred, split_labels = load_sarima_data(dataset_name)

    dataset_results = []

    for run_num in range(1, NUM_RUNS + 1):
        try:
            result = run_single_experiment(
                dataset_name, run_num, model_type,
                serie, linear_pred, split_labels,
            )
            dataset_results.append(result)
            save_results_to_csv(result, output_file)

            print(
                f"  {model_type:12s} | {dataset_name:12s} "
                f"| Run {run_num:02d}/{NUM_RUNS} "
                f"| Test MSE: {result['mse_test']:.4e} "
                f"| Time: {result['training_time_sec']:.3f}s "
                f"| RSS: {result['peak_memory_mb']:.2f}MB "
                f"| Py: {result['peak_python_memory_mb']:.2f}MB"
            )

        except Exception as exc:
            print(f"  {model_type} | {dataset_name} | Run {run_num:02d} FAILED: {exc}")

    # ── Per-dataset summary ───────────────────────────────────
    if dataset_results:
        mses  = [r["mse_test"]              for r in dataset_results if not np.isnan(r["mse_test"])]
        rmses = [r["rmse_test"]             for r in dataset_results if not np.isnan(r["rmse_test"])]
        times = [r["training_time_sec"]     for r in dataset_results]
        mem_r = [r["peak_memory_mb"]        for r in dataset_results]
        mem_p = [r["peak_python_memory_mb"] for r in dataset_results]

        print(f"\n  [{model_type}/{dataset_name}] Summary ({len(dataset_results)} runs):")
        if mses:
            print(f"    Test MSE        : {np.mean(mses):.4e}  ±  {np.std(mses):.4e}")
            print(f"    Test RMSE       : {np.mean(rmses):.4e}  ±  {np.std(rmses):.4e}")
        print(f"    Training time   : {np.mean(times):.4f}s  ±  {np.std(times):.4f}s")
        print(f"    Peak RSS        : {np.mean(mem_r):.2f}MB  ±  {np.std(mem_r):.2f}MB")
        print(f"    Peak Py-heap    : {np.mean(mem_p):.2f}MB  ±  {np.std(mem_p):.2f}MB\n")

    return dataset_results


print("Experiment-pipeline helpers loaded.")

Experiment-pipeline helpers loaded.


In [10]:
# ============================================================
# CELL 10 – MAIN EXECUTION
# ============================================================
#
# Runs all four hybrid model types across all ten datasets.
#
# Grid:
#   4 model types  x  10 datasets  x  15 runs  =  600 result rows
#
# SVR runs are deterministic (all 15 rows per dataset are identical);
# MLP runs use set_seed(42 + run_num) so they vary across runs.
# ============================================================

MODEL_TYPES = ["ARIMA_SVR", "ARIMA_MLP", "SARIMA_SVR", "SARIMA_MLP"]
OUTPUT_FILE = "results/hybrid_models_results.csv"

# Remove any stale results file to ensure a clean, reproducible run
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)
    print(f"Removed existing {OUTPUT_FILE}")

print("=" * 70)
print("Starting Hybrid Models Experiments")
print(f"  Model types  : {MODEL_TYPES}")
print(f"  Datasets     : {len(DATASETS)}")
print(f"  Runs / combo : {NUM_RUNS}")
print(f"  Total runs   : {len(MODEL_TYPES) * len(DATASETS) * NUM_RUNS}")
print(f"  Output       : {OUTPUT_FILE}")
print("=" * 70 + "\n")

all_results = []

for model_type in MODEL_TYPES:
    print(f"\n{'#' * 70}")
    print(f"  MODEL TYPE: {model_type}")
    print(f"{'#' * 70}")

    for dataset_name in DATASETS:
        print(f"\n  --- Dataset: {dataset_name} ---")
        results = run_dataset_experiments(dataset_name, model_type, OUTPUT_FILE)
        all_results.extend(results)

print(f"\n{'#' * 70}")
print("ALL EXPERIMENTS COMPLETED")
print(f"  Total successful runs : {len(all_results)}")
print(f"  Results saved to      : {OUTPUT_FILE}")
print(f"{'#' * 70}")

# Quick sanity check
if os.path.exists(OUTPUT_FILE):
    df_check = pd.read_csv(OUTPUT_FILE)
    print(f"\nRows in CSV    : {len(df_check)}")
    print(f"Methods found  : {sorted(df_check['method'].unique())}")
    print(f"Datasets found : {sorted(df_check['dataset'].unique())}")
    print(f"\nColumn list    :\n{list(df_check.columns)}")
    print(f"\nSample rows (first 4):")
    print(df_check.head(4).to_string(index=False))

Starting Hybrid Models Experiments
  Model types  : ['ARIMA_SVR', 'ARIMA_MLP', 'SARIMA_SVR', 'SARIMA_MLP']
  Datasets     : 10
  Runs / combo : 15
  Total runs   : 600
  Output       : results/hybrid_models_results.csv


######################################################################
  MODEL TYPE: ARIMA_SVR
######################################################################

  --- Dataset: CARSALES ---
[ARIMA/CARSALES] n=108 | Train=52 | Val=36 | Test=20
  ARIMA_SVR    | CARSALES     | Run 01/15 | Test MSE: 2.6139e-02 | Time: 0.003s | RSS: 0.23MB | Py: 0.03MB
  ARIMA_SVR    | CARSALES     | Run 02/15 | Test MSE: 2.6139e-02 | Time: 0.001s | RSS: 0.00MB | Py: 0.01MB
  ARIMA_SVR    | CARSALES     | Run 03/15 | Test MSE: 2.6139e-02 | Time: 0.001s | RSS: 0.00MB | Py: 0.01MB
  ARIMA_SVR    | CARSALES     | Run 04/15 | Test MSE: 2.6139e-02 | Time: 0.001s | RSS: 0.00MB | Py: 0.01MB
  ARIMA_SVR    | CARSALES     | Run 05/15 | Test MSE: 2.6139e-02 | Time: 0.001s | RSS: 0.00MB | Py: 0.0